--- Day 16: Reindeer Maze ---
It's time again for the Reindeer Olympics! This year, the big event is the Reindeer Maze, where the Reindeer compete for the lowest score.

You and The Historians arrive to search for the Chief right as the event is about to start. It wouldn't hurt to watch a little, right?

The Reindeer start on the Start Tile (marked S) facing East and need to reach the End Tile (marked E). They can move forward one tile at a time (increasing their score by 1 point), but never into a wall (#). They can also rotate clockwise or counterclockwise 90 degrees at a time (increasing their score by 1000 points).

To figure out the best place to sit, you start by grabbing a map (your puzzle input) from a nearby kiosk. For example:

###############
#.......#....E#
#.#.###.#.###.#
#.....#.#...#.#
#.###.#####.#.#
#.#.#.......#.#
#.#.#####.###.#
#...........#.#
###.#.#####.#.#
#...#.....#.#.#
#.#.#.###.#.#.#
#.....#...#.#.#
#.###.#.#.#.#.#
#S..#.....#...#
###############
There are many paths through this maze, but taking any of the best paths would incur a score of only 7036. This can be achieved by taking a total of 36 steps forward and turning 90 degrees a total of 7 times:


###############
#.......#....E#
#.#.###.#.###^#
#.....#.#...#^#
#.###.#####.#^#
#.#.#.......#^#
#.#.#####.###^#
#..>>>>>>>>v#^#
###^#.#####v#^#
#>>^#.....#v#^#
#^#.#.###.#v#^#
#^....#...#v#^#
#^###.#.#.#v#^#
#S..#.....#>>^#
###############
Here's a second example:

#################
#...#...#...#..E#
#.#.#.#.#.#.#.#.#
#.#.#.#...#...#.#
#.#.#.#.###.#.#.#
#...#.#.#.....#.#
#.#.#.#.#.#####.#
#.#...#.#.#.....#
#.#.#####.#.###.#
#.#.#.......#...#
#.#.###.#####.###
#.#.#...#.....#.#
#.#.#.#####.###.#
#.#.#.........#.#
#.#.#.#########.#
#S#.............#
#################
In this maze, the best paths cost 11048 points; following one such path would look like this:

#################
#...#...#...#..E#
#.#.#.#.#.#.#.#^#
#.#.#.#...#...#^#
#.#.#.#.###.#.#^#
#>>v#.#.#.....#^#
#^#v#.#.#.#####^#
#^#v..#.#.#>>>>^#
#^#v#####.#^###.#
#^#v#..>>>>^#...#
#^#v###^#####.###
#^#v#>>^#.....#.#
#^#v#^#####.###.#
#^#v#^........#.#
#^#v#^#########.#
#S#>>^..........#
#################
Note that the path shown above includes one 90 degree turn as the very first move, rotating the Reindeer from facing East to facing North.

Analyze your map carefully. What is the lowest score a Reindeer could possibly get?

Your puzzle answer was 122492.

--- Part Two ---
Now that you know what the best paths look like, you can figure out the best spot to sit.

Every non-wall tile (S, ., or E) is equipped with places to sit along the edges of the tile. While determining which of these tiles would be the best spot to sit depends on a whole bunch of factors (how comfortable the seats are, how far away the bathrooms are, whether there's a pillar blocking your view, etc.), the most important factor is whether the tile is on one of the best paths through the maze. If you sit somewhere else, you'd miss all the action!

So, you'll need to determine which tiles are part of any best path through the maze, including the S and E tiles.

In the first example, there are 45 tiles (marked O) that are part of at least one of the various best paths through the maze:

###############
#.......#....O#
#.#.###.#.###O#
#.....#.#...#O#
#.###.#####.#O#
#.#.#.......#O#
#.#.#####.###O#
#..OOOOOOOOO#O#
###O#O#####O#O#
#OOO#O....#O#O#
#O#O#O###.#O#O#
#OOOOO#...#O#O#
#O###.#.#.#O#O#
#O..#.....#OOO#
###############
In the second example, there are 64 tiles that are part of at least one of the best paths:

#################
#...#...#...#..O#
#.#.#.#.#.#.#.#O#
#.#.#.#...#...#O#
#.#.#.#.###.#.#O#
#OOO#.#.#.....#O#
#O#O#.#.#.#####O#
#O#O..#.#.#OOOOO#
#O#O#####.#O###O#
#O#O#..OOOOO#OOO#
#O#O###O#####O###
#O#O#OOO#..OOO#.#
#O#O#O#####O###.#
#O#O#OOOOOOO..#.#
#O#O#O#########.#
#O#OOO..........#
#################
Analyze your map further. How many tiles are part of at least one of the best paths through the maze?

Your puzzle answer was 520.

from copy import deepcopy # Actually not needed
from queue import Queue

import sys
sys.setrecursionlimit(10**5)

In [2]:
def read_input(file_name):
    f = []
    with open(file_name, 'r') as file:
        for row in file:
            f.append(list(row.strip()))
    return f

In [3]:
def draw_field(field):
    for r in field:
        print(''.join(r), end='\n')            

In [4]:
def get_next_coordinates(path):

    (x, y) = path[-1]
    result = []
    if x-1 >= 0:
        result.append((x-1, y))
    if y+1 < len(field[0]):
        result.append((x, y+1))
    if x+1 < len(field):
        result.append((x+1, y))
    if y-1 >= 0:
        result.append((x, y-1))

    return result
    

In [5]:
def generate_next_paths():

    increment = False
    #print(f'Call of generate_next_paths. Initially the paths are:')
    #print(paths)
    
    for path in paths:

        #print(f'Processing path {path}')
        # Get what are the coordinates of the possible continuation of the path (all four directions)
        next_steps = get_next_coordinates(path)
        #print('Next steps are:')
        #print(next_steps)
        #print('\n\n')
    
        # If not move backwards and if no wall, consider the new path made from the existing one with added the continuation coordinate
        for step in next_steps:

            current_path = deepcopy(path)

            if step not in current_path:
                current_path.append(step)
                
                if step not in path and field[step[0]][step[1]] != '#' and field[step[0]][step[1]] != 'E' and current_path not in paths:
                    paths.append(current_path)
                    increment = True
    
                if field[step[0]][step[1]] == 'E' and current_path not in final_paths:
                    final_paths.append(current_path)
                    increment = True
        
        paths.remove(path)

    return increment


In [6]:
def calculate_pats(s_coordinates):

    increment = True
    itteration = 0

    while increment:
        itteration += 1
        #print(f'Execution of itteration {itteration}')
        increment = generate_next_paths()
        if itteration % 10 == 0:
            print(len(paths))
            print(paths[-1])    


In [7]:
def find_s_e(field):
    for r in range(len(field)):
        for c in range(len(field[0])):
            
            if field[r][c] == 'S':
                s = (r, c)
            
            if field[r][c] == 'E':
                e = (r, c)
    
    return (s, e)

In [8]:
def calc_score(path):
    score = 0
    previous_2 = (path[0][0], path[0][1] - 1)
    previous_1 = path[0]
    for c in range(1,len(path)):
        #print(f'path[c] = {path[c]}, previous_1 = {previous_1}, previous_2 = {previous_2}', end='')
        if (path[c][0] - previous_1[0] == previous_1[0] - previous_2[0]) and (path[c][1] - previous_1[1] == previous_1[1] - previous_2[1]):
            score += 1
            #print(' Same Direction')
            previous_2 = previous_1
            previous_1 = path[c]
        else:
            score += 1001
            #print(' CHANGE Direction')
            previous_2 = previous_1
            previous_1 = path[c]            
    return score
        

In [9]:
def encode_coordinates(p_coordinates):
    return p_coordinates[0] * 1000 + p_coordinates[1]

def decode_coordinates(p_code):
    return (p_code // 1000, p_code % 1000)

In [10]:
def get_next_positions(code):

    (x, y) = decode_coordinates(code)
    
    result = []
    
    if x-1 >= 0 and field[x-1][y] in ['.', 'E']:
        result.append((x-1, y, '^'))
    if y+1 < len(field[0]) and field[x][y+1] in ['.', 'E']: 
        result.append((x, y+1, '>')) 
    if x+1 < len(field) and field[x+1][y] in ['.', 'E']:
        result.append((x+1, y, 'v'))
    if y-1 >= 0 and field[x][y-1] in ['.', 'E']:
        result.append((x, y-1, '<'))

    return result

In [11]:
def get_next_positions_backwards(code):

    (x, y) = decode_coordinates(code)
    
    result = []
    
    if x-1 >= 0 and field[x-1][y] in ['.', 'S']:
        result.append((x-1, y, 'v'))
    if y+1 < len(field[0]) and field[x][y+1] in ['.', 'S']: 
        result.append((x, y+1, '<')) 
    if x+1 < len(field) and field[x+1][y] in ['.', 'S']:
        result.append((x+1, y, '^'))
    if y-1 >= 0 and field[x][y-1] in ['.', 'S']:
        result.append((x, y-1, '>'))

    return result

In [76]:
def calc_position_scores():
    iteration = True
    ctr = 0
    while iteration:

        iteration = False
    
        initial_position_scores_keys = tuple(position_scores.keys())
        #print('\nInitial position scores: ', initial_position_scores)
        ctr += 1
        #print(f'ctr={ctr} for {len(initial_position_scores_keys)} keys', end='  ')
        
        #For each already caolculated position we will try to extend with one step
        for k in initial_position_scores_keys:

            #print(f'Calculating k = {k}')
            
            if position_scores[k][2] == 1:

                #print(f'position_scores[{k}] = {position_scores[k]}')
                
                # Take the possible next positions
                new_pos = get_next_positions(k)
                #print(f'new_pos = {new_pos}')
           
                for next_code in new_pos:
    
                    code = encode_coordinates((next_code[0], next_code[1]))
                    direction_sequence = position_scores[k][0] + next_code[2]
                    
                    next_code_path = []
                    
                    for pt in position_scores[k][3]:
                        p_temp = deepcopy(pt)
                        #print(f'pt before = {p_temp}')
                        p_temp.append((next_code[0], next_code[1]))
                        next_code_path.append(p_temp)
                        #print(f'pt after = {p_temp}')
                        #print(f'next_code_path = {next_code_path}')
                        
                    
                    if position_scores[k][0][-1] != next_code[2]:
                        score = position_scores[k][1] + 1001
                    else:
                        score = position_scores[k][1] + 1   
    
                    if (position_scores.get(code) is None):
                        position_scores[code] = [direction_sequence, score, 1, next_code_path]
                        #print(f'Adding position_scores[{code}] = {position_scores[code]}')
                        iteration = True

                    elif position_scores[code][1] > score:
                        position_scores[code] = [direction_sequence, score, 1, next_code_path]
                        #print(f'Adding position_scores[{code}] = {position_scores[code]}')                        
                        iteration = True         

                    #elif position_scores[code][1] == score:
                    #    position_scores[code][3].append(next_code_path)
                    #    print(f'Adding position_scores[{code}] = {position_scores[code]}')                        
                    #    iteration = True 
                        

            position_scores[k][2] = 0
                    

In [77]:
def calc_position_scores_backwards():
    iteration = True
    while iteration:

        iteration = False

        #initial_position_scores = deepcopy(position_scores_backwards)
        initial_position_scores_keys = tuple(position_scores_backwards.keys())
        #print('\nInitial position scores: ', initial_position_scores)
        
        #For each already caolculated position we will try to extend with one step
        for k in initial_position_scores_keys:

            #print(f'Calculating k = {k}')
            
            if position_scores_backwards[k][2] == 1:

                #print(f'position_scores[k] = {position_scores[k]}')
                
                # Take the possible next positions
                new_pos = get_next_positions_backwards(k)
                #print(f'new_pos = {new_pos}')
           
                for next_code in new_pos:
    
                    code = encode_coordinates((next_code[0], next_code[1]))
                    direction_sequence = next_code[2] + position_scores_backwards[k][0]
                    
                    next_code_path = []
                    
                    for pt in position_scores_backwards[k][3]:
                        p_temp = deepcopy(pt)
                        #print(f'pt before = {p_temp}')
                        p_temp.append((next_code[0], next_code[1]))
                        next_code_path.append(p_temp)
                        #print(f'pt after = {p_temp}')
                        #print(f'next_code_path = {next_code_path}')
                        
                    try:
                        if position_scores_backwards[k][0] and position_scores_backwards[k][0][0] != next_code[2]:
                            score = position_scores_backwards[k][1] + 1001
                        else:
                            score = position_scores_backwards[k][1] + 1   
                    except IndexError:
                        print(f'Exxeption for position_scores_backwards[{k}] = {position_scores_backwards[k]}')
                        
    
                    if (position_scores_backwards.get(code) is None):
                        position_scores_backwards[code] = [direction_sequence, score, 1, next_code_path]
                        #print(f'Adding position_scores[{code}] = {position_scores[code]}')
                        iteration = True

                    elif position_scores_backwards[code][1] > score:
                        position_scores_backwards[code] = [direction_sequence, score, 1, next_code_path]
                        #print(f'Adding position_scores[{code}] = {position_scores[code]}')                        
                        iteration = True         

                    #elif position_scores_backwards[code][1] == score:
                    #    position_scores_backwards[code][3].append(next_code_path)
                    #    print(f'Adding position_scores[{code}] = {position_scores_backwards[code]}')                        
                    #    iteration = True 
                        

            position_scores_backwards[k][2] = 0

In [69]:
def fill_paths_dict(p_field, p_start, p_end, print_freq):
    directions = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    end_paths = []

    max_x = len(p_field) - 1
    max_y = len(p_field[0]) -1

    paths_queue = Queue()
    paths_queue.put([s_coordinates])

    itteration = 0

    while not paths_queue.empty():

        itteration += 1

        if itteration % print_freq == 0:
            print(f'On itteration {itteration} the queue has {paths_queue.qsize()} paths')
        
        current_path = paths_queue.get()
        last_position = current_path[-1]

        if last_position == p_end:
            end_paths.append(current_path)

        for direction in directions:
            next_position = (last_position[0] + direction[0], last_position[1] + direction[1])
            
            if (next_position[0] >= 0 and 
                next_position[0] <= max_x and
                next_position[1] >= 0 and 
                next_position[1] <= max_y and 
                p_field[next_position[0]][next_position[1]] != '#' and
                next_position not in current_path # avoid cycles
               ):

                    next_path = list(current_path)
                    next_path.append(next_position)
                    if len(next_path) <= 500:
                        paths_queue.put(next_path)
                
    return end_paths                

In [70]:
def calc_bidirectional_position_score(coordinates):
    c = encode_coordinates(coordinates)
    #print(position_scores_backwards[c])
    #print(position_scores[c])
    b_part = position_scores_backwards[c][3][0]
    #print('\n')
    b_part.reverse()
    #print(b_part)
    try:
        path =  position_scores[c][3][0] + b_part[1:]
    except KeyError:
        print('EXCEPTION ON: \n' + position_scores)
    #print(path)
    return calc_score(path)

In [79]:
#field = read_input('input_example_1.txt')
#field = read_input('input_example_2.txt')
field = read_input('input.txt')

draw_field(field)

(s_coordinates, e_coordinates) = find_s_e(field)

real_s_coordinates = s_coordinates
#s_coordinates = (s_coordinates[0], s_coordinates[1] - 1)
print(s_coordinates)

#end_paths_list = fill_paths_dict(field, s_coordinates, e_coordinates, 1000000)

position_scores = {encode_coordinates(s_coordinates): ['>', 0, 1, [[s_coordinates]]]}
calc_position_scores()

print(f'calc_position_scores() is executed')
#print(position_scores)

position_scores_backwards = {encode_coordinates(e_coordinates): ['', 0, 1, [[e_coordinates]]]}
calc_position_scores_backwards()

print(f'calc_position_scores_backwards() is executed')

#fill_paths_dict()

ctr = 0
target_score = calc_bidirectional_position_score((e_coordinates[0],e_coordinates[1]))
print(target_score)
for x in range(len(field)):
    for y in range(len(field[0])):
        if field[x][y] in ('.', 'S', 'E'):
            score = calc_bidirectional_position_score((x, y))
            #print(f'field[{x}][{y}] = {score}')
            if score == target_score:
                ctr += 1
            

print(ctr)

#############################################################################################################################################
#.....#.........#.#.........#.........#.......#.................................#...#.....#.......#.......#.....#.......#..................E#
#.###.#.#.#####.#.#.#####.###.#####.#.#.#####.#.#########.#.###.#.###.#####.#.#.#.###.#.###.#.#.#.#.#.###.#.###.#.#.###.#.#########.#####.###
#...#...#...#.....#.....#.#...#.....#.#.......................#.................#.....#.....#.#...#.#.#...#.#.#...#.#...#.#...#.........#...#
###.#.###.#.#.#########.#.#.###.#####.#.#.#.#.#.###.#.###.###.###.#.#.#.#.#.#.#######.###.###.#####.#.#####.#.#####.###.#.#.#.###.#.###.###.#
#...#...#.........................#...#.#.#...#...#.#.......#.#...#...#.#.#.#.....#.....#...#.....#.#.............#...#.#.#.#...#.#...#...#.#
#.#####.#.#.#####.#.###.#####.###.#.###.#.#.###.#.#.#.###.###.#.#.#.###.#.#.#.#.#.#.#.#.#####.#.#.#.###########.#####.###.#.###.#.###.#####.#
#.#...